# channel_no_conc Batch Workflow

批量入口 notebook：扫描一个目录下第一层的主配置 JSON，展开其中引用的全部模板，并逐个运行 `channel_no_conc` compare。


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start=None):
    cwd = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "examples" / "neuron_compare" / "channel_no_conc" / "workflows" / "workflow_api.py").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


REPO_ROOT = find_repo_root()
CHANNEL_NO_CONC_ROOT = REPO_ROOT / "examples" / "neuron_compare" / "channel_no_conc"
WORKFLOWS_ROOT = CHANNEL_NO_CONC_ROOT / "workflows"
if str(WORKFLOWS_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKFLOWS_ROOT))

import workflow_api
import brainstate
brainstate.environ.set(precision=64)


## Parameters

| 参数 | 含义 | 常见取值 |
|---|---|---|
| `config_dir` | 批量扫描目录；只扫描第一层 `*.json`，每个 config 会跑完自己声明的全部模板。 | `CHANNEL_NO_CONC_ROOT / "configs"` |
| `summary_dir` | 本次 batch run 的输出根目录。 | `None` 自动生成为 `results/batch_runs/YYMMDD_HHMMSS/`；或手动指定路径 |
| `plot_cases` | 是否在每个 config 的模板子目录里生成图产物。 | `True` / `False` |


In [2]:
config_dir = CHANNEL_NO_CONC_ROOT / "configs" / "ma20_goc"  # 批量扫描目录；每个 config 会跑完自己声明的全部模板
summary_dir = None  # 本次 batch run 的输出根目录；None 时自动使用 YYMMDD_HHMMSS 命名
plot_cases = True # 是否在每个 config 的模板子目录里生成图产物


## Discover Configs And Prepare Batch Directory


In [3]:
config_records = workflow_api.discover_batch_configs(config_dir)
batch_run_id = workflow_api.make_batch_run_id()
resolved_summary_dir = workflow_api.default_batch_run_output_dir(
    batch_run_id=batch_run_id,
    summary_dir=summary_dir,
)
configs_out_dir = resolved_summary_dir / "configs"
configs_out_dir.mkdir(parents=True, exist_ok=True)

preview_df = pd.DataFrame(
    [
        {
            "config_name": row["config_name"],
            "config_path": str(row["config_path"]),
            "n_templates": row["n_templates"],
            "template_names": ", ".join(row["template_names"]),
            "config_out_dir": str(configs_out_dir / row["config_name"]),
        }
        for row in config_records
    ]
)
print("batch_run_id:", batch_run_id)
print("resolved_summary_dir:", resolved_summary_dir)
display(preview_df)


batch_run_id: 260604_225333
resolved_summary_dir: /home/swl/braincell-ion_dyn/examples/neuron_compare/channel_no_conc/results/batch_runs/260604_225333


,config_name,config_path,n_templates,template_names,config_out_dir
0,cahva_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
1,cav1p2_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
2,cav1p3_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
3,cav2p3_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
4,cav3p1_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
5,hcn1_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
6,hcn2_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
7,kca1p1_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
8,kca2p2_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...
9,kca3p1_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,3,"vinit_celsius, dc, ac",/home/swl/braincell-ion_dyn/examples/neuron_co...


## Run Each Config


In [4]:
config_run_infos = []
for record in config_records:
    config_out_dir = configs_out_dir / str(record["config_name"])
    run_info = workflow_api.run_notebook_config_workflow(
        config_path=record["config_path"],
        out_dir=config_out_dir,
        plot=plot_cases,
        expand_only=False,
        raise_on_failure=False,
    )
    config_run_infos.append(run_info)

batch_summary = workflow_api.write_batch_summary_artifacts(
    config_dir=config_dir,
    summary_dir=resolved_summary_dir,
    batch_run_id=batch_run_id,
    config_records=config_records,
    config_run_infos=config_run_infos,
    plot_cases=plot_cases,
)
print("batch_manifest:", batch_summary["manifest_path"])
print("batch_observable_summary_csv:", batch_summary["batch_observable_summary_path"])
print("batch_observable_summary_json:", batch_summary["batch_observable_summary_json_path"])
print("batch_failures_csv:", batch_summary["batch_failures_path"])


--No graphics will be displayed.


batch_manifest: /home/swl/braincell-ion_dyn/examples/neuron_compare/channel_no_conc/results/batch_runs/260604_225333/batch_manifest.json
batch_observable_summary_csv: /home/swl/braincell-ion_dyn/examples/neuron_compare/channel_no_conc/results/batch_runs/260604_225333/batch_observable_summary.csv
batch_observable_summary_json: /home/swl/braincell-ion_dyn/examples/neuron_compare/channel_no_conc/results/batch_runs/260604_225333/batch_observable_summary.json
batch_failures_csv: /home/swl/braincell-ion_dyn/examples/neuron_compare/channel_no_conc/results/batch_runs/260604_225333/batch_failures.csv


## Load Batch Summary Tables


In [5]:
runs_df = pd.read_csv(batch_summary["config_runs_path"]).sort_values(["batch_status", "config_name"]).reset_index(drop=True)
observables_df = pd.read_csv(batch_summary["batch_observable_summary_path"]).sort_values(["observable", "config_name"]).reset_index(drop=True)
failures_df = pd.read_csv(batch_summary["batch_failures_path"]).reset_index(drop=True)

display(observables_df)
display(failures_df)


,config_name,config_path,batch_status,observable,n_cases,mae_mean,rmse_mean,max_abs_max,rel_mae_pct_mean
0,cahva_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,current.ix,10,2.681551e-15,4.312209e-15,9.158646e-14,3.592672e-10
1,cav1p2_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,current.ix,10,4.219999e-16,7.716922e-16,1.494594e-14,2.276707e-10
2,cav1p3_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,current.ix,10,1.089012e-16,1.449281e-16,1.549488e-15,4.419016e-10
3,cav2p3_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,current.ix,10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
4,cav3p1_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,current.ix,10,1.015591e-05,1.653115e-05,9.853243e-05,1.909539e-01
...,...,...,...,...,...,...,...,...,...
74,km_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,voltage,10,1.208108e-10,1.401119e-10,2.034156e-09,1.606664e-10
75,kv1p1_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,voltage,10,3.138279e-11,3.954841e-11,3.954170e-10,4.290824e-11
76,kv3p4_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,voltage,10,9.350821e-11,1.078994e-10,1.515374e-09,1.232391e-10
77,kv4p3_ma20_goc,/home/swl/braincell-ion_dyn/examples/neuron_co...,ok,voltage,10,1.175781e-10,1.365662e-10,1.940322e-09,1.545407e-10


,config_name,config_path,batch_status,template_name,template_path,out_dir,n_failed_cases,error_message


如果某个 config 需要继续做 template / case 级钻取，进入该 batch run 目录下的 `configs/<config_name>/`，再用 [workflow.ipynb](/home/swl/braincell/examples/neuron_compare/channel_no_conc/workflows/workflow.ipynb) 对单个 config 继续分析。
